# Phase 3: Data Preprocessing

In this phase we prepare the dataset for machine learning.

The following steps will be performed:

1. Train-Test Split
2. Feature Encoding
3. Feature Scaling
4. Handling Class Imbalance using SMOTE
5. Preparing final datasets for model training

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [10]:
from imblearn.over_sampling import SMOTE

In [3]:
# Load dataset
df = pd.read_csv("../data/student_dataset.csv", sep=";")

# Show first rows
df.head()

,Marital status,Application mode,Application order,Course,Daytime/evening attendance\t,Previous qualification,Previous qualification (grade),Nacionality,Mother's qualification,Father's qualification,...,Curricular units 2nd sem (credited),Curricular units 2nd sem (enrolled),Curricular units 2nd sem (evaluations),Curricular units 2nd sem (approved),Curricular units 2nd sem (grade),Curricular units 2nd sem (without evaluations),Unemployment rate,Inflation rate,GDP,Target
0,1,17,5,171,1,1,122.0,1,19,12,...,0,0,0,0,0.000000,0,10.8,1.4,1.74,Dropout
1,1,15,1,9254,1,1,160.0,1,1,3,...,0,6,6,6,13.666667,0,13.9,-0.3,0.79,Graduate
2,1,1,5,9070,1,1,122.0,1,37,37,...,0,6,0,0,0.000000,0,10.8,1.4,1.74,Dropout
3,1,17,2,9773,1,1,122.0,1,38,37,...,0,6,10,5,12.400000,0,9.4,-0.8,-3.12,Graduate
4,2,39,1,8014,0,1,100.0,1,37,38,...,0,6,6,6,13.000000,0,13.9,-0.3,0.79,Graduate


## Create Binary Target Variable

The original dataset contains three target classes:

• Graduate  
• Enrolled  
• Dropout  

Since this project focuses on predicting whether a student will drop out, the problem is converted into a **binary classification task**.

Mapping used:

Dropout → 1  
Graduate / Enrolled → 0

This new column is named **Dropout_Binary**.

In [4]:
# Create binary dropout target
df["Dropout_Binary"] = df["Target"].apply(lambda x: 1 if x == "Dropout" else 0)

# Check distribution
df["Dropout_Binary"].value_counts()

Dropout_Binary
0    3003
1    1421
Name: count, dtype: int64

## Separate Features and Target

Machine learning models require input variables (features) and an output variable (target).

The dataset is split into:

**X (features)** → all independent variables used for prediction  
**y (target)** → the dropout label

The columns `Target` and `Dropout_Binary` are removed from the feature set to prevent data leakage.

In [5]:
# Separate features and target
X = df.drop(columns=["Target", "Dropout_Binary"])
y = df["Dropout_Binary"]

print("Feature shape:", X.shape)
print("Target shape:", y.shape)

Feature shape: (4424, 36)
Target shape: (4424,)


## Train–Test Split

The dataset is divided into training and testing sets.

Training data is used to train the model, while testing data is used to evaluate how well the model performs on unseen data.

An 80/20 split is used:

• 80% for training  
• 20% for testing  

Stratification is applied to preserve the original dropout class distribution in both datasets.

In [6]:
# Train-Test Split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

X_train: (3539, 36)
X_test: (885, 36)
y_train: (3539,)
y_test: (885,)


In [7]:
print("Training class distribution:")
print(y_train.value_counts(normalize=True))

print("\nTesting class distribution:")
print(y_test.value_counts(normalize=True))

Training class distribution:
Dropout_Binary
0    0.678723
1    0.321277
Name: proportion, dtype: float64

Testing class distribution:
Dropout_Binary
0    0.679096
1    0.320904
Name: proportion, dtype: float64


In [8]:
# Identify categorical and numerical columns

categorical_cols = X_train.select_dtypes(include=["object"]).columns
numerical_cols = X_train.select_dtypes(include=["int64", "float64"]).columns

print("Categorical columns:", list(categorical_cols))
print("\nNumber of categorical columns:", len(categorical_cols))

print("\nNumerical columns:", len(numerical_cols))

Categorical columns: []

Number of categorical columns: 0

Numerical columns: 36


## Feature Scaling

Machine learning algorithms often perform better when features are on a similar scale.

StandardScaler is used to standardize the features so that they have:

• Mean = 0  
• Standard Deviation = 1  

The scaler is fitted on the training data and then applied to both training and testing sets to avoid data leakage.

In [9]:
# Feature scaling

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Scaled training shape:", X_train_scaled.shape)
print("Scaled testing shape:", X_test_scaled.shape)

Scaled training shape: (3539, 36)
Scaled testing shape: (885, 36)


## Handle Class Imbalance Using SMOTE

The dataset contains more non-dropout students than dropout students.

This class imbalance can cause machine learning models to favor the majority class.

SMOTE (Synthetic Minority Oversampling Technique) is used to generate synthetic samples of the minority class.

Important rule followed:

SMOTE is applied **only to the training data**, while the testing data remains unchanged.

In [11]:
# Apply SMOTE to training data only
smote = SMOTE(random_state=42)

X_train_smote, y_train_smote = smote.fit_resample(X_train_scaled, y_train)

print("X_train_smote shape:", X_train_smote.shape)
print("y_train_smote shape:", y_train_smote.shape)

X_train_smote shape: (4804, 36)
y_train_smote shape: (4804,)


In [12]:
print("Balanced class distribution after SMOTE:")
print(pd.Series(y_train_smote).value_counts())

Balanced class distribution after SMOTE:
Dropout_Binary
1    2402
0    2402
Name: count, dtype: int64


## Convert Arrays Back to DataFrames

After scaling and applying SMOTE, the resulting data is stored as NumPy arrays.

To preserve feature names and improve interpretability, the arrays are converted back into pandas DataFrames.

This allows easier analysis and enables feature importance visualization in later stages.

In [13]:
# Convert scaled arrays back to DataFrames

X_train_smote = pd.DataFrame(X_train_smote, columns=X_train.columns)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X_test.columns)

print("Training dataframe shape:", X_train_smote.shape)
print("Testing dataframe shape:", X_test_scaled.shape)

Training dataframe shape: (4804, 36)
Testing dataframe shape: (885, 36)


## Save Processed Datasets

After completing preprocessing, the final training and testing datasets are saved to disk.

Saving the processed datasets allows later stages of the project, such as model training and evaluation, to directly load clean and prepared data without repeating preprocessing steps.

The following datasets are saved:

- X_train_smote
- y_train_smote
- X_test_scaled
- y_test

In [15]:
# Save processed datasets

X_train_smote.to_csv("../data/processed/X_train_smote.csv", index=False)
y_train_smote.to_csv("../data/processed/y_train_smote.csv", index=False)

X_test_scaled.to_csv("../data/processed/X_test_scaled.csv", index=False)
y_test.to_csv("../data/processed/y_test.csv", index=False)

print("Processed datasets saved successfully.")

Processed datasets saved successfully.
